In [2]:
import asyncio
import os
import sys
from pathlib import Path
import sys
import threading
import time

import numpy as np
import pandas as pd
import yfinance as yf
from dotenv import load_dotenv

# Make direct script execution work by ensuring the project root is on sys.path.
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

from data.db import statements, client

# bargain hunting

In [32]:
# table: metadata + fundamental data

db = client.init_async_db()

async with db.begin() as conn:
    metadata_df = await statements.get_metadata(conn)
    fundamental_df = await statements.get_fundamental_data(conn)

metadata_df = metadata_df[["stock_id", "ticker", "short_name"]]
fundamental_df = fundamental_df[["stock_id", "retrieve_at", "trailing_pe", "total_cash", "total_debt", "free_cashflow"]]

data_df = pd.merge(
    metadata_df,
    fundamental_df,
    how="left",
    on=["stock_id"]
)

In [33]:
display(data_df)

,stock_id,ticker,short_name,retrieve_at,trailing_pe,total_cash,total_debt,free_cashflow
0,1,AALI.JK,Astra Agro Lestari Tbk.,2026-07-17 17:28:22.704144,7.9171,4703853740032,0,1.803771e+12
1,1,AALI.JK,Astra Agro Lestari Tbk.,2026-07-17 17:50:40.411695,7.9171,4703853740032,0,1.803771e+12
2,1,AALI.JK,Astra Agro Lestari Tbk.,2026-07-31 15:26:22.644401,7.0822,4703853740032,0,1.803771e+12
3,1,AALI.JK,Astra Agro Lestari Tbk.,2026-07-31 08:28:53.859218,7.1079,4703853740032,0,1.803771e+12
4,2,ABBA.JK,Mahaka Media Tbk.,2026-07-17 17:28:22.704144,None,18275401728,209548951552,1.030826e+10
...,...,...,...,...,...,...,...,...
395,99,BRMS.JK,Bumi Resources Minerals Tbk.,2026-07-31 08:28:53.859218,84.0841,105391744,260616752,-3.569710e+07
396,100,BRNA.JK,Berlina Tbk,2026-07-17 17:28:22.704144,99.8336,37077868544,894152409088,1.327044e+10
397,100,BRNA.JK,Berlina Tbk,2026-07-17 17:50:40.411695,99.8336,37077868544,894152409088,1.327044e+10
398,100,BRNA.JK,Berlina Tbk,2026-07-31 15:26:22.644401,99.0017,37077868544,894152409088,1.327044e+10


In [34]:
data_df["net_cash"] = data_df["total_cash"] - data_df["total_debt"]

In [36]:
display(
    data_df[["stock_id", "retrieve_at", "total_cash", "total_debt", "net_cash"]]
)

,stock_id,retrieve_at,total_cash,total_debt,net_cash
0,1,2026-07-17 17:28:22.704144,4703853740032,0,4703853740032
1,1,2026-07-17 17:50:40.411695,4703853740032,0,4703853740032
2,1,2026-07-31 15:26:22.644401,4703853740032,0,4703853740032
3,1,2026-07-31 08:28:53.859218,4703853740032,0,4703853740032
4,2,2026-07-17 17:28:22.704144,18275401728,209548951552,-191273549824
...,...,...,...,...,...
395,99,2026-07-31 08:28:53.859218,105391744,260616752,-155225008
396,100,2026-07-17 17:28:22.704144,37077868544,894152409088,-857074540544
397,100,2026-07-17 17:50:40.411695,37077868544,894152409088,-857074540544
398,100,2026-07-31 15:26:22.644401,37077868544,894152409088,-857074540544
